In [1]:
import csv
from getpass import getpass
from google import genai

api_key = getpass("Enter your Gemini API key: ")

client = genai.Client(api_key=api_key)

GEMINI_MODEL = "gemini-3.5-flash"

print("Gemini client created successfully.")

Enter your Gemini API key:  ········


Gemini client created successfully.


In [2]:
def load_properties(filename):
    properties = []

    try:
        with open(
            filename,
            mode="r",
            newline="",
            encoding="utf-8"
        ) as file:

            reader = csv.DictReader(file)

            for row in reader:
                property_info = {
                    "property_name": row["property_name"].strip(),
                    "monthly_rent": float(row["monthly_rent"]),
                    "income_multiplier": float(
                        row["income_multiplier"]
                    ),
                    "min_credit_score": int(
                        row["min_credit_score"]
                    )
                }

                properties.append(property_info)

        print(f"{len(properties)} properties loaded successfully.")

        return properties

    except FileNotFoundError:
        print(f"Error: The file '{filename}' was not found.")
        return []

    except KeyError:
        print("Error: The CSV column names are incorrect.")
        return []

    except ValueError:
        print("Error: One or more CSV values are invalid.")
        return []

In [3]:
properties = load_properties("properties.csv")

3 properties loaded successfully.


In [4]:
def display_properties(property_list):
    print("\n" + "=" * 72)
    print("RENTSMART AVAILABLE PROPERTIES")
    print("=" * 72)

    for index, property_info in enumerate(
        property_list,
        start=1
    ):
        print(f"{index}. {property_info['property_name']}")

        print(
            f"   Monthly Rent: "
            f"${property_info['monthly_rent']:,.2f}"
        )

        print(
            f"   Income Requirement: "
            f"{property_info['income_multiplier']}x monthly rent"
        )

        print(
            f"   Minimum Credit Score: "
            f"{property_info['min_credit_score']}"
        )

        print("-" * 72)

In [5]:
def select_property(property_list):
    while True:
        try:
            choice = int(
                input("Select a property number: ")
            )

            if 1 <= choice <= len(property_list):
                return property_list[choice - 1]

            print(
                f"Please enter a number between 1 and "
                f"{len(property_list)}."
            )

        except ValueError:
            print(
                "Invalid input. Please enter a whole number."
            )

In [6]:
def get_monthly_income():
    while True:
        try:
            monthly_income = float(
                input("Enter your gross monthly income: $")
            )

            if monthly_income > 0:
                return monthly_income

            print(
                "Monthly income must be greater than $0."
            )

        except ValueError:
            print(
                "Invalid input. Please enter numbers only."
            )

In [7]:
def get_credit_score():
    while True:
        try:
            credit_score = int(
                input("Enter your credit score (300-850): ")
            )

            if 300 <= credit_score <= 850:
                return credit_score

            print(
                "Credit score must be between 300 and 850."
            )

        except ValueError:
            print(
                "Invalid input. Please enter a whole number."
            )

In [8]:
def evaluate_applicant(
    selected_property,
    monthly_income,
    credit_score
):
    monthly_rent = selected_property["monthly_rent"]

    income_multiplier = (
        selected_property["income_multiplier"]
    )

    minimum_credit_score = (
        selected_property["min_credit_score"]
    )

    required_income = (
        monthly_rent * income_multiplier
    )

    income_pass = (
        monthly_income >= required_income
    )

    credit_pass = (
        credit_score >= minimum_credit_score
    )

    if income_pass and credit_pass:
        status = "Likely Qualified"

    elif income_pass or credit_pass:
        status = "Risky"

    else:
        status = "Not Qualified"

    assessment = {
        "required_income": required_income,
        "income_pass": income_pass,
        "credit_pass": credit_pass,
        "status": status
    }

    return assessment

In [9]:
def generate_report(
    selected_property,
    monthly_income,
    credit_score,
    assessment
):
    income_result = (
        "PASSED"
        if assessment["income_pass"]
        else "FAILED"
    )

    credit_result = (
        "PASSED"
        if assessment["credit_pass"]
        else "FAILED"
    )

    print("\n" + "=" * 72)
    print("RENTSMART RENTAL QUALIFICATION REPORT")
    print("=" * 72)

    print("\nPROPERTY INFORMATION")
    print("-" * 72)

    print(
        f"Property Name:              "
        f"{selected_property['property_name']}"
    )

    print(
        f"Monthly Rent:               "
        f"${selected_property['monthly_rent']:,.2f}"
    )

    print(
        f"Income Multiplier:          "
        f"{selected_property['income_multiplier']}"
        f"x monthly rent"
    )

    print(
        f"Minimum Credit Score:       "
        f"{selected_property['min_credit_score']}"
    )

    print("\nAPPLICANT INFORMATION")
    print("-" * 72)

    print(
        f"Monthly Income:             "
        f"${monthly_income:,.2f}"
    )

    print(
        f"Credit Score:               "
        f"{credit_score}"
    )

    print("\nQUALIFICATION RESULTS")
    print("-" * 72)

    print(
        f"Required Monthly Income:    "
        f"${assessment['required_income']:,.2f}"
    )

    print(
        f"Income Requirement:         "
        f"{income_result}"
    )

    print(
        f"Credit Requirement:         "
        f"{credit_result}"
    )

    print(
        f"Final Status:               "
        f"{assessment['status']}"
    )

In [10]:
def get_gemini_explanation(
    selected_property,
    monthly_income,
    credit_score,
    assessment
):
    prompt = f"""
    Explain this preliminary rental qualification result
    in plain language.

    Property:
    {selected_property['property_name']}

    Monthly rent:
    ${selected_property['monthly_rent']:,.2f}

    Required monthly income:
    ${assessment['required_income']:,.2f}

    Applicant monthly income:
    ${monthly_income:,.2f}

    Minimum credit score:
    {selected_property['min_credit_score']}

    Applicant credit score:
    {credit_score}

    Income requirement passed:
    {assessment['income_pass']}

    Credit requirement passed:
    {assessment['credit_pass']}

    Final qualification status:
    {assessment['status']}

    Explain why the applicant received this status.

    Keep the explanation under 120 words.

    Include one practical recommendation.

    Clearly state that this is a preliminary assessment
    and does not guarantee rental approval.
    """

    try:
        response = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt
        )

        return response.text

    except Exception as error:
        return (
            "The Gemini explanation is currently unavailable. "
            "The Python qualification report is still valid.\n"
            f"Technical error: {error}"
        )

In [11]:
def run_rentsmart():
    if not properties:
        print(
            "RentSmart cannot start because no "
            "property data was loaded."
        )
        return

    print("\n" + "=" * 72)
    print("WELCOME TO RENTSMART")
    print("Rental Qualification Decision Support System")
    print("=" * 72)

    display_properties(properties)

    selected_property = select_property(properties)

    print(
        f"\nYou selected: "
        f"{selected_property['property_name']}"
    )

    monthly_income = get_monthly_income()

    credit_score = get_credit_score()

    assessment = evaluate_applicant(
        selected_property,
        monthly_income,
        credit_score
    )

    generate_report(
        selected_property,
        monthly_income,
        credit_score,
        assessment
    )

    print("\nGEMINI AI EXPLANATION")
    print("-" * 72)

    gemini_explanation = get_gemini_explanation(
        selected_property,
        monthly_income,
        credit_score,
        assessment
    )

    print(gemini_explanation)

    print("\nDISCLAIMER")
    print("-" * 72)

    print(
        "This is a preliminary decision-support "
        "assessment and does not guarantee final "
        "rental approval."
    )

    print("=" * 72)

In [12]:
run_rentsmart()


WELCOME TO RENTSMART
Rental Qualification Decision Support System

RENTSMART AVAILABLE PROPERTIES
1. Riata Place
   Monthly Rent: $1,400.00
   Income Requirement: 3.0x monthly rent
   Minimum Credit Score: 600
------------------------------------------------------------------------
2. Lakeview Apartments
   Monthly Rent: $1,200.00
   Income Requirement: 2.5x monthly rent
   Minimum Credit Score: 580
------------------------------------------------------------------------
3. City Center Lofts
   Monthly Rent: $1,800.00
   Income Requirement: 3.0x monthly rent
   Minimum Credit Score: 650
------------------------------------------------------------------------


Select a property number:  1



You selected: Riata Place


Enter your gross monthly income: $ 4000
Enter your credit score (300-850):  620



RENTSMART RENTAL QUALIFICATION REPORT

PROPERTY INFORMATION
------------------------------------------------------------------------
Property Name:              Riata Place
Monthly Rent:               $1,400.00
Income Multiplier:          3.0x monthly rent
Minimum Credit Score:       600

APPLICANT INFORMATION
------------------------------------------------------------------------
Monthly Income:             $4,000.00
Credit Score:               620

QUALIFICATION RESULTS
------------------------------------------------------------------------
Required Monthly Income:    $4,200.00
Income Requirement:         FAILED
Credit Requirement:         PASSED
Final Status:               Risky

GEMINI AI EXPLANATION
------------------------------------------------------------------------
The Gemini explanation is currently unavailable. The Python qualification report is still valid.
Technical error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high d